In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

# Step 1: load the features made in notebook 2
features = pd.read_csv("data/customer_features.csv", index_col="Customer ID")
X = features.drop(columns="churn")   # inputs
y = features["churn"]                # what we predict

# Step 2: split 80% train / 20% test (stratify keeps the same churn % in both)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# Step 3: the three models to compare
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=8,
                                            min_samples_leaf=5, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

# Step 4: train each model and measure it on the test set
results = []
for name, model in models.items():
    # pipeline = log transform -> scaling -> model (same steps every time)
    pipe = Pipeline([("log", FunctionTransformer(np.log1p)),
                     ("scale", StandardScaler()),
                     ("model", model)])
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, prob),
        "cv_roc_auc": cv_auc,
    })
    print(name, "confusion matrix:")
    print(confusion_matrix(y_test, pred))

pd.DataFrame(results).round(3)

Logistic Regression confusion matrix:
[[267 117]
 [110 180]]
Random Forest confusion matrix:
[[257 127]
 [117 173]]
Gradient Boosting confusion matrix:
[[257 127]
 [117 173]]


,model,accuracy,precision,recall,f1,roc_auc,cv_roc_auc
0,Logistic Regression,0.663,0.606,0.621,0.613,0.717,0.743
1,Random Forest,0.638,0.577,0.597,0.586,0.702,0.736
2,Gradient Boosting,0.638,0.577,0.597,0.586,0.699,0.731


In [2]:
import joblib, os

# Rebuild the best model (Logistic Regression) and train it
best = Pipeline([("log", FunctionTransformer(np.log1p)),
                 ("scale", StandardScaler()),
                 ("model", LogisticRegression(max_iter=1000))])
best.fit(X_train, y_train)

# Save the whole pipeline and the feature order for the API
os.makedirs("model", exist_ok=True)
joblib.dump({"pipeline": best, "features": list(X.columns)}, "model/baseline_model.joblib")

# Which features push churn up or down? (bigger number = stronger effect)
coefs = best.named_steps["model"].coef_[0]
print(dict(zip(X.columns, coefs.round(2))))

{'recency_days': np.float64(0.09), 'frequency': np.float64(-1.47), 'monetary': np.float64(0.69), 'tenure_days': np.float64(0.17), 'total_items': np.float64(-0.27), 'avg_order_value': np.float64(-0.45)}


In [3]:
pd.DataFrame(results).round(3)

,model,accuracy,precision,recall,f1,roc_auc,cv_roc_auc
0,Logistic Regression,0.663,0.606,0.621,0.613,0.717,0.743
1,Random Forest,0.638,0.577,0.597,0.586,0.702,0.736
2,Gradient Boosting,0.638,0.577,0.597,0.586,0.699,0.731


In [4]:
import joblib
# Should print exactly 6 feature names
print(joblib.load("model/baseline_model.joblib")["features"])

['recency_days', 'frequency', 'monetary', 'tenure_days', 'total_items', 'avg_order_value']


In [5]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

# Probabilities for the TRAINING data, predicted fairly (the test set stays untouched)
prob = cross_val_predict(best, X_train, y_train, cv=5, method="predict_proba")[:, 1]

# Try different thresholds: flag as churn if probability >= threshold
for t in [0.3, 0.4, 0.5, 0.6]:
    pred = (prob >= t).astype(int)
    print(t,
          "precision", round(precision_score(y_train, pred), 3),
          "recall", round(recall_score(y_train, pred), 3),
          "f1", round(f1_score(y_train, pred), 3),
          "flagged", pred.sum(), "of", len(pred))

0.3 precision 0.543 recall 0.891 f1 0.675 flagged 1902 of 2696
0.4 precision 0.583 recall 0.774 f1 0.665 flagged 1538 of 2696
0.5 precision 0.631 recall 0.624 f1 0.628 flagged 1145 of 2696
0.6 precision 0.646 recall 0.433 f1 0.519 flagged 777 of 2696


In [6]:
print(joblib.load("model/baseline_model.joblib")["features"])

['recency_days', 'frequency', 'monetary', 'tenure_days', 'total_items', 'avg_order_value']


In [7]:
# Final check on the test set with the chosen threshold
threshold = 0.4
test_prob = best.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= threshold).astype(int)

print("precision", round(precision_score(y_test, test_pred), 3),
      "recall", round(recall_score(y_test, test_pred), 3),
      "f1", round(f1_score(y_test, test_pred), 3))
print(confusion_matrix(y_test, test_pred))

# Save model + threshold + feature order together for the API
joblib.dump({"pipeline": best, "features": list(X.columns), "threshold": threshold},
            "model/churn_model.joblib")

precision 0.564 recall 0.748 f1 0.643
[[216 168]
 [ 73 217]]


['model/churn_model.joblib']